In [1]:
!pip install lapjv
!pip install igraph
!pip install leidenalg

import numpy as np
import scipy
from scipy import sparse
from lapjv import lapjv
import time
from itertools import combinations
import scipy.io
import pandas as pd
import csv
from google.colab import drive
from collections import defaultdict
from os import cpu_count
import re
from collections import defaultdict
import matplotlib.pyplot as plt
import igraph as ig
import leidenalg as la
import random
from typing import List



  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for lapjv: filename=lapjv-1.3.27-cp312-cp312-linux_x86_64.whl size=142135 sha256=cdd3fc084d6a05b20ffd5eb96ec0971a7d2d127d5ae342104ba8ac22ed944133
  Stored in directory: /root/.cache/pip/wheels/8a/b5/65/3cbb79ec56625bf795b818545064022bc30460016a31ab9b5f
Successfully built lapjv
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 36.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 39.6 MB/s eta 0:00:00


In [9]:
# All helper methods used for data processing

def has_duplicate_2d(array):
    seen = set()
    for row in array:
        for elem in row:
            if elem in seen:
                return True
            seen.add(elem)
    return False

def count_singleton_lists(d):
    total_values = len(d)
    count_singleton = sum(1 for v in d.values() if isinstance(v, list) and len(v) == 1)
    ratio = count_singleton / total_values if total_values > 0 else 0
    return count_singleton, ratio

def generate_unique_pairs(arr):
    return list(combinations(arr, 2))

def merge_single_element_sublists(lst):
    singles = []
    result = []
    for sub in lst:
        if len(sub) == 1:
            singles.extend(sub)
        else:
            result.append(sub)
    if singles:
        result.append(singles)
    return result

def barycenter_matrices(n, m):
    k = len(n)
    cum_n = np.cumsum(n)
    cum_n = [0] + cum_n.tolist()

    PP = np.zeros((cum_n[-1], cum_n[-1]));
    P = np.zeros((cum_n[-1], cum_n[-1]));
    for i in range(0, k):
        n_u_i = n[i]-m[i]
        P[cum_n[i]+m[i]:cum_n[i+1], cum_n[i]+m[i]:cum_n[i+1]] = np.ones((n_u_i, n_u_i))/n_u_i;
        if m[i] == 0:
            PP[cum_n[i]:cum_n[i+1], cum_n[i]:cum_n[i+1]] =  P[cum_n[i]:cum_n[i+1], cum_n[i]:cum_n[i+1]]
        else:
            PP[cum_n[i]:cum_n[i]+m[i], cum_n[i]:cum_n[i]+m[i]] = np.identity(m[i]);
            PP[(cum_n[i]+m[i]):cum_n[i+1], (cum_n[i]+m[i]):cum_n[i+1]] = np.ones((n_u_i, n_u_i))/n_u_i;
    return P, PP

def group_indices(arr):
  unique_vals = np.unique(arr)
  index_groups = {}

  for val in unique_vals:
    indices = np.where(arr == val)[0].tolist()
    index_groups[val] = indices
  return index_groups

def generate_array_comm(n, k):
    base_value = n // k
    remainder = n % k
    result = [base_value] * k

    for i in range(remainder):
        result[i] += 1

    return result

def new_generate_array_comm(l, x):
    total = sum(l)
    ratio = x / total
    l2 = [min(int(size * ratio), size) for size in l]
    for i in range(len(l)):
        if l2[i] == l[i]:
            if l2[i] > 0:
                l2[i] -= 1
            else:
                l2[i] += 1
    current_sum = sum(l2)
    difference = x - current_sum

    fractional_parts = [(size * ratio) - int(size * ratio) for size in l]
    indices_sorted = sorted(
        range(len(l)),
        key=lambda i: (fractional_parts[i], -l[i]),
        reverse=True
    )

    for i in indices_sorted:
        if difference <= 0:
            break
        if abs(l2[i] + 1 - l[i]) >= 1 and l2[i] < l[i]:
            l2[i] += 1
            difference -= 1

    return l2

def direct_sum(A, B):
    return np.block([
        [A, np.zeros((A.shape[0], B.shape[1]))],
        [np.zeros((B.shape[0], A.shape[1])), B]
    ])

def label_to_numbers(labels):
    label_mapping = {}
    numbered_labels = []

    for label in labels:
        if label not in label_mapping:
            label_mapping[label] = len(label_mapping)
        numbered_labels.append(label_mapping[label])

    return numbered_labels, label_mapping

def relabel_alphebatically(mapping, label_list):
    sorted_keys = sorted(list(mapping.keys()))
    alphebatical_labels = [mapping[key] for key in sorted_keys if key in mapping]
    perm = np.argsort(alphebatical_labels)
    new_mapping = dict(zip(sorted_keys, list(range(0,len(sorted_keys)))))
    new_label_list = [int(perm[i]) for i in label_list]
    return new_label_list, new_mapping

def right_perm(mapping, length):
    mapped_comm = map_keys(mapping, list(range(0,length)))
    ind = np.argsort(mapped_comm)
    ind2 = np.argsort(ind)
    ind = [int(x) for x in ind]
    ind2 = [int(x) for x in ind2]
    return mapped_comm, ind, ind2

def merge_ones(lst):
    i = 0
    while i < len(lst):
        if lst[i] == 1:
            if i == 0:
                lst[i + 1] += lst[i]
            elif i == len(lst) - 1:
                lst[i - 1] += lst[i]
            else:
                lst[i + 1] += lst[i]
            lst.pop(i)
            i = max(0, i - 1)
        else:
            i += 1
    return lst

def acc(corr, perm, num_seeds, total):
    return ((sum(1 for a, b in zip(perm, corr) if a == b))-num_seeds)/(total-num_seeds)

def permute_communities(A, B, D):
    groups = []
    current = 0
    for size in B:
        groups.append(A[current:current + size])
        current += size

    result = []
    for key in sorted(D.keys()):
        for b_index in D[key]:
            result.extend(groups[b_index])

    return result

def get_var_name(variable):
     for name, value in globals().items():
        if value is variable:
            return name

def label_to_numbers(labels):
    label_mapping = {}
    numbered_labels = []

    for label in labels:
        if label not in label_mapping:
            label_mapping[label] = len(label_mapping)
        numbered_labels.append(label_mapping[label])

    return numbered_labels, label_mapping

def find_all_occurrences(input_list):
    unique_indices = defaultdict(list)
    for index, value in enumerate(input_list):
        unique_indices[value].append(index)
    return dict(unique_indices)

def map_correspondence(list1, list2):
    if len(list1) != len(list2):
        raise ValueError("Both lists must have the same length")

    correspondence = {}
    visited = set()
    for key, value in zip(list1, list2):
        if key not in correspondence:
            correspondence[key] = set()
        if value not in visited:
            visited.add(value)
            correspondence[key].add(value)

    return {k: sorted(v) for k, v in correspondence.items()}

def group_array_by_ind_element(arr, ind):
  grouped_data = {}
  for entry in arr:
    element = entry[ind]
    if element not in grouped_data:
      grouped_data[element] = []
    grouped_data[element].append(entry)
  return grouped_data

def outer_product(X):
    return np.dot(X.T, X)

def read_file(to_read):
    matrix = np.empty((0,992))
    first = True
    pData = pd.read_csv("pData.csv", index_col=-1)
    with open(to_read, 'r') as file:
        reader = csv.reader(file)
        for row in reader:
            if first:
                row_num = np.array([float(num) for num in row[0].split()])
                indices = np.vstack([list(pData['index ']), row_num])
                first = False
            else:
                row_num = np.array([float(num) for num in row[0].split()])
                matrix = np.vstack([matrix, row_num])
    return matrix

def map_keys(d, Arr):
    element_to_key = {}
    for key, values in d.items():
        for val in values:
            element_to_key[val] = key
    return [element_to_key[num] for num in Arr]

def permute_array(arr, perm):
    result = [0] * len(arr)

    for i in range(len(arr)):
        result[i] = arr[perm[i]]

    return result

def get_comm_size(arr):
    sizes = []
    count = 1

    for i in range(1, len(arr)):
        if arr[i] == arr[i - 1]:
            count += 1
        else:
            sizes.append(count)
            count = 1

    sizes.append(count)
    return sizes

def generate_seed_in_comm(comm_size, seed_num):
    seed_in_comm = [0] * len(comm_size)

    if seed_num < len(comm_size):
        seed_in_comm[:seed_num] = [1] * seed_num
    else:
        seed_in_comm = [1] * len(comm_size)
        current_sum = len(comm_size)
        max_possible = sum(size - 1 for size in comm_size)

        if seed_num == 0:
            seed_in_comm = [0] * len(comm_size)

        remaining = seed_num - current_sum
        indices = [i for i in range(len(comm_size)) if comm_size[i] > 1]

        while remaining > 0 and indices:
            per_element = max(1, remaining // len(indices))

            for i in indices.copy():
                alloc = min(per_element, comm_size[i] - 1 - seed_in_comm[i], remaining)
                seed_in_comm[i] += alloc
                remaining -= alloc

                if seed_in_comm[i] >= comm_size[i] - 1:
                    indices.remove(i)


    final_perm = []
    prev = 0
    for index, value in enumerate(seed_in_comm):
        final_perm = final_perm + list(range(prev,prev+seed_in_comm[index]))
        final_perm = final_perm + [int(x) for x in list(np.random.permutation(range(prev+seed_in_comm[index], prev+comm_size[index])))]
        prev = prev + comm_size[index]
    final_perm = [int(x) for x in final_perm]

    non_comm_perm = list(range(0,sum(seed_in_comm))) + [int(x) for x in list(np.random.permutation(range(sum(seed_in_comm), sum(comm_size))))]

    return seed_in_comm, final_perm, non_comm_perm


def shuffle_percentage(arr, percentage):
    n = len(arr)
    count = int(n * percentage / 100)
    if count == 0:
        return arr
    indices_to_shuffle = random.sample(range(n), count)
    elements_to_shuffle = [arr[i] for i in indices_to_shuffle]
    random.shuffle(elements_to_shuffle)
    for i, idx in enumerate(indices_to_shuffle):
        arr[idx] = elements_to_shuffle[i]
    return arr

def leiden_communities_to_labels(communities):
    n = max(max(comm) for comm in communities) + 1
    labels = [None] * n
    for community_idx, nodes in enumerate(communities):
        for node in nodes:
            labels[node] = community_idx
    return labels

In [10]:
# block_SGM function
# n -- a vec of partition sizes
# m -- a vec of seeds per partition
# It is assumed that the first m(i) vertices of partiton i
# correspond respectively to the first m(i) vertices of partition i of B,


def block_SGM(A, B, n, m, max_iter, tol):

    P, PP = barycenter_matrices(n, m);
    k = len(n);
    patience = max_iter;
    iter = 0;
    go_on = 1;
    cum_n = np.cumsum(n);
    cum_n = [0] + cum_n.tolist();
    u = [n[i] - m[i] for i in range(k)];


    while iter < max_iter and go_on == 1:

        old_P = PP;
        order = range(0,k)

        for ind in range(k):
            j = order[ind]
            grad_j = np.zeros((n[j], n[j]))
            for i in range(k):
                if i == j:
                    grad_j = grad_j + A[cum_n[i]:cum_n[i+1],cum_n[i]:cum_n[i+1]] \
                    @ PP[cum_n[i]:cum_n[i+1], cum_n[i]:cum_n[i+1]] \
                    @ (B[cum_n[i]:cum_n[i+1],cum_n[i]:cum_n[i+1]]).T \
                    + (A[cum_n[i]:cum_n[i+1],cum_n[i]:cum_n[i+1]]).T \
                    @ PP[cum_n[i]:cum_n[i+1], cum_n[i]:cum_n[i+1]] @ (B[cum_n[i]:cum_n[i+1],cum_n[i]:cum_n[i+1]])

                else:
                    grad_j = grad_j + 2*((A[cum_n[i]:cum_n[i+1],cum_n[j]:cum_n[j+1]]).T\
                    @ PP[cum_n[i]:cum_n[i+1], cum_n[i]:cum_n[i+1]]\
                    @ B[cum_n[i]:cum_n[i+1], cum_n[j]:cum_n[j+1]])

            #make round a flag
            cost_matrix = -np.round(grad_j[m[j]:, m[j]:] , 5)
            col_ind, _, _ = lapjv(cost_matrix)
            Tj = np.eye(n[j]-m[j])[col_ind, :]
            Tj = direct_sum(np.eye(m[j]), Tj)


            R = PP.copy()
            R[cum_n[j]:cum_n[j+1], cum_n[j]:cum_n[j+1]] = Tj

            T1 = (A @ PP @ B.T @ PP.T).diagonal().sum()
            T2 = (A @ PP @ B.T @ R.T).diagonal().sum()
            T3 = (A @ R @ B.T @ PP.T).diagonal().sum()
            T4 = (A @ R @ B.T @ R.T).diagonal().sum()

            if (2 * (T1 - T2 - T3 + T4)) == 0:
                PP = PP;
            else:
                beta = (-T2 - T3 + 2 * T4) / (2 * (T1 - T2 - T3 + T4))

                Qj = Tj.copy()

                P_new_potential = beta * PP[cum_n[j]:cum_n[j+1], cum_n[j]:cum_n[j+1]] + (1-beta)*Qj

                PP_new_potential = PP.copy()
                PP_new_potential[cum_n[j]:cum_n[j+1], cum_n[j]:cum_n[j+1]] = P_new_potential

                PP_new_potential_beta0 = PP.copy()
                PP_new_potential_beta0[cum_n[j]:cum_n[j+1], cum_n[j]:cum_n[j+1]] = Qj


                obj_beta0 =  -(A @ PP_new_potential_beta0 @ B.T @ PP_new_potential_beta0.T).diagonal().sum()

                obj_potential = -(A @ PP_new_potential @ B.T @ PP_new_potential.T).diagonal().sum()
                obj_now = -(A @ PP @ B.T @ PP.T).diagonal().sum()

                #print(beta)
                if beta > 1 or beta < 0:
                    if obj_beta0 < obj_now:
                        PP = PP_new_potential_beta0
                        #print("step 0")
                    #else:
                    #    print("step 1")

                if beta <=1 and beta >= 0:
                    if obj_potential < obj_now and obj_potential < obj_beta0:
                        PP = PP_new_potential
                        #print("step beta")
                    elif obj_potential > obj_beta0 and obj_now > obj_beta0:
                        PP = PP_new_potential_beta0
                        #print("step 0")
                    #else:
                    #    print("step 1")

        iter += 1

        s = np.sum(np.abs(old_P - PP))

        if s < 1 - tol:
            #print("it ends with s < 1 - tol")
            go_on = 0

    corr = [None] * cum_n[-1]

    # make corr vector for each group
    for i in range(k):

            P[cum_n[i]:cum_n[i+1], cum_n[i]:cum_n[i+1]] = np.round(PP[cum_n[i]:cum_n[i+1], cum_n[i]:cum_n[i+1]],5)
            col_ind, _, _ = lapjv(-P[cum_n[i]:cum_n[i+1], cum_n[i]:cum_n[i+1]])

            for j in range(n[i]):
                if j < m[i]:
                    corr[cum_n[i] + j] = cum_n[i] + j
                else:
                    corr[cum_n[i] + j] = cum_n[i] + col_ind[j]

    #print("This one uses "+str(iter)+"iterations")

    return corr

In [ ]:
# Regular Wiki Experiment
wiki_adj = scipy.io.loadmat('wiki_adj.mat')
wiki_labels = scipy.io.loadmat('wiki_labels.mat')

graphA = wiki_adj['G_EN_Adj']
graphB = wiki_adj['G_FR_Adj']
wiki_labels = wiki_labels['wiki_labels'][0]
graphA2 = graphA.copy()

comm_len2 = [np.sum(wiki_labels == i) for i in range(1, 7)]
avg_arr = []

for i in range(0, 100):
      print("-------------------------------------------------------")
      print("Current number of seeds:" + str(i))
      avg = 0
      for j in range(0,10):
            print("iteration:"+str(j))
            seeds_in_comm2 = generate_array_comm(i, 6)

            cum_comm = np.cumsum(comm_len2)
            cum_comm = [0] + cum_comm.tolist()
            perm = np.array(range(0,cum_comm[-1]))


            for i in range(0, len(seeds_in_comm2)):
                # Calculate the slice size
                slice_size = cum_comm[i+1] - (cum_comm[i] + seeds_in_comm2[i])

                # Generate permutation with the correct slice size
                permuted_indices = np.random.permutation(slice_size)

                # Shift permuted indices to the correct range
                shifted_indices = permuted_indices + (cum_comm[i] + seeds_in_comm2[i])

                # Assign to the slice of perm
                perm[(cum_comm[i]+seeds_in_comm2[i]):cum_comm[i+1]] = shifted_indices

            graphA2[:] = graphA[perm][:, perm]
            #print(perm)
            corr = block_SGM(graphA2, graphB, comm_len2, seeds_in_comm2, 20 , 1-10**(-6));
            num_seeds = np.sum(seeds_in_comm2)
            avg = avg+ ((np.sum(perm == corr)-num_seeds)/(1382-num_seeds))
            print((np.sum(perm == corr)-num_seeds)/(1382-num_seeds))

      avg_arr = avg_arr + [avg/10]
      print("avg result for this number of seeds is"+str(avg/10))
      print("End for this number of seeds" + str(i))
      print("-------------------------------------------------------")

In [ ]:
#Wiki experiments with shuffling

wiki_adj = scipy.io.loadmat('wiki_adj.mat')
wiki_labels = scipy.io.loadmat('wiki_labels.mat')['wiki_labels'][0]
indices = wiki_labels.argsort()

graphA = wiki_adj['G_EN_Adj'][indices][:, indices]
graphB = wiki_adj['G_FR_Adj'][indices][:, indices]

graphA2 = graphA.copy()

comm_len2 = [np.sum(wiki_labels == i) for i in range(1, 7)]
avg_arr = []

for percentage in [0,5,10,20,30,40,50,60,70,80,90,100]:
      shuffled_perm = shuffle_percentage(list(range(0,1382)), percentage)
      print("-------------------------------------------------------")
      print("Current percentage of shuffling:" + str(percentage))
      avg = 0
      for i in [5,50,150,250,382,500]:
            print("-------------------------------------------------------")
            print("Current number of seeds:" + str(i))
            avg = 0

            seeds_in_comm2 = generate_array_comm(i, 6)

            for j in range(0,5):
                  cum_comm = np.cumsum(comm_len2)
                  cum_comm = [0] + cum_comm.tolist()
                  perm = np.array(range(0,cum_comm[-1]))

                  for i in range(0, len(seeds_in_comm2)):
                      # Calculate the slice size
                      slice_size = cum_comm[i+1] - (cum_comm[i] + seeds_in_comm2[i])

                      # Generate permutation with the correct slice size
                      permuted_indices = np.random.permutation(slice_size)

                      # Shift permuted indices to the correct range
                      shifted_indices = permuted_indices + (cum_comm[i] + seeds_in_comm2[i])

                      # Assign to the slice of perm
                      perm[(cum_comm[i]+seeds_in_comm2[i]):cum_comm[i+1]] = shifted_indices

                  graphA3 = graphA.copy()
                  graphA3[:] = graphA3[shuffled_perm][:, shuffled_perm]
                  graphA2[:] = graphA3[perm][:, perm]
                  #print(perm)
                  corr = block_SGM(graphA2, graphB, comm_len2, seeds_in_comm2, 20, 1-10**(-6));
                  num_seeds = np.sum(seeds_in_comm2)
                  avg = avg+ ((np.sum(perm == corr)-num_seeds)/(1382-num_seeds))
                  print((np.sum(perm == corr)-num_seeds)/(1382-num_seeds))

            avg_arr = avg_arr + [avg/5]
            print("avg result for this number of seeds is"+str(avg/3))
            print("End for this number of seeds" + str(i))
            print("-------------------------------------------------------")

In [ ]:
# C.elegans experiments

elegans = scipy.io.loadmat('elegansGraph.mat')
elegans_labels = scipy.io.loadmat('celegans_labels.mat')['cel_labels']
graphA = elegans['Achem'].toarray()
graphB = elegans['Agap'].toarray()

labels_vec = elegans_labels['cel_labels'].flatten()
elegans_indices = labels_vec.argsort()
comm_len = [np.sum(labels_vec == i) for i in range(1, 4)]
cum_comm = np.cumsum(comm_len)
cum_comm = [0] + cum_comm.tolist()


graphA = elegans['Achem'][elegans_indices][:, elegans_indices]
graphB = elegans['Agap'][elegans_indices][:, elegans_indices]

acc = [[]]
acc2 = {}
for n_of_seeds in [0,1,5,10,20,50,75,100,150,200]:
        print("-------------------------------------------------------")
        print("Current number of seeds:" + str(n_of_seeds))
        n_seeds = n_of_seeds
        seeds_in_comm = new_generate_array_comm(comm_len, n_of_seeds)
        acc_j = []
        for j in range(100):
            perm = np.array(range(0,cum_comm[-1]))
            for i in range(0, len(seeds_in_comm)):
                slice_size = cum_comm[i+1] - (cum_comm[i] + seeds_in_comm[i])
                permuted_indices = np.random.permutation(slice_size)
                shifted_indices = permuted_indices + (cum_comm[i] + seeds_in_comm[i])
                perm[(cum_comm[i]+seeds_in_comm[i]):cum_comm[i+1]] = shifted_indices
            graphA3 = graphA.copy()
            graphA2[:] = graphA3[perm][:, perm]

            corr = block_SGM(graphA2, graphB, comm_len, seeds_in_comm, 15, 1-10**(-6));
            print(str((np.sum(perm == corr)-n_of_seeds)/(279-n_of_seeds)))
            acc_j = acc_j + [(np.sum(perm == corr)-n_of_seeds)/(279-n_of_seeds)]
        print("-------------------------------------------------------")
        acc = acc + [acc_j]
        acc2[n_of_seeds] =  acc_j
        corr2 = block_SGM(graphA, graphB, comm_len, seeds_in_comm, 15, 1-10**(-6));
        print((np.sum(np.arange(0,279) == corr2)-n_seeds)/(279-n_seeds))

In [ ]:
import networkx as nx
from networkx.algorithms.community import girvan_newman

wiki_adj = scipy.io.loadmat('wiki_adj.mat')
graphA = wiki_adj['G_EN_Adj']
graphB = wiki_adj['G_FR_Adj']
graphA2 = graphA.copy()

graph = ig.Graph.Adjacency(graphB.tolist(), mode="undirected")
la_partition_wiki = list(la.find_partition(graph, la.ModularityVertexPartition))


In [ ]:
wiki_adj = scipy.io.loadmat('wiki_adj.mat')

graphA = wiki_adj['G_EN_Adj']
graphB = wiki_adj['G_FR_Adj']
graphA2 = graphA.copy()

graph = ig.Graph.Adjacency(graphB, mode="undirected")
N = graph.vcount()
n_runs = 100

all_leiden_wiki = []

for i in range(n_runs):
    part = list(la.find_partition(graph, la.ModularityVertexPartition, seed=i))
    all_leiden_wiki.append(part)

np.save('100runs_leiden_wiki_with_singletons.npy', np.array(all_leiden_wiki, dtype=object), allow_pickle=True)

all_wiki_here = np.load('100runs_leiden_wiki_with_singletons.npy', allow_pickle=True)

all_wiki_labels_with_singleton = []
all_wiki_labels_without_singleton = []

for i in range(100):
    all_wiki_labels_with_singleton.append(leiden_communities_to_labels(all_wiki_here[i]))

for i in range(100):
    all_wiki_labels_without_singleton.append(leiden_communities_to_labels(merge_single_element_sublists(all_wiki_here[i])))


np.save('100runs_leiden_wiki_with_singletons_labels.npy', np.array(all_wiki_labels_with_singleton, dtype=object), allow_pickle=True)
np.save('100runs_leiden_wiki_without_singletons_labels.npy', np.array(all_wiki_labels_without_singleton, dtype=object), allow_pickle=True)

In [6]:
import os
import numpy as np

# Below are experiments of table 2&3 for CESSNA, the rest of non-CESSNA experiments are in Experiments_table2&3_non-CESSNA.ipynb

base_path = '/content/'
rhos = [round(1.0 - 0.1*i, 1) for i in range(11)]
idxs = range(9, -1, -1)

# Loading matrices
matrices3 = {}
for rho in rhos:
    for idx in idxs:
        for label in ['A', 'B']:
            filename = f'Experiments3_data/P3_matrix_rho_{rho}_idx_{idx}_{label}.npy'
            file_path = os.path.join(base_path, filename)
            if os.path.isfile(file_path):
                matrices3[(rho, idx, label)] = np.load(file_path)

# 1. Correlation = 0.3

r = 0.3
for i in range(10):
    graphA = matrices3[(r, i, "A")]
    graphB = matrices3[(r, i, "B")]

    graphA2 = graphA.copy()
    comm_len = [100,100,100]
    this_one_0_3 = []
    this_num_iter = []

    for n_seeds in [1, 5, 10, 20, 50, 75, 100, 150, 200]:
          seed_i = []
          seed_i_iter = []
          seeds_in_comm2 = new_generate_array_comm(comm_len, n_seeds)
          for j in range(0,10):
                        cum_comm = np.cumsum(comm_len)
                        cum_comm = [0] + cum_comm.tolist()
                        perm = np.array(range(0,cum_comm[-1]))


                        for i in range(0, len(seeds_in_comm2)):
                            slice_size = cum_comm[i+1] - (cum_comm[i] + seeds_in_comm2[i])
                            permuted_indices = np.random.permutation(slice_size)
                            shifted_indices = permuted_indices + (cum_comm[i] + seeds_in_comm2[i])
                            perm[(cum_comm[i]+seeds_in_comm2[i]):cum_comm[i+1]] = shifted_indices

                        graphA3 = graphA.copy()
                        graphA2[:] = graphA3[perm][:, perm]
                        corr = block_SGM(graphA2, graphB, comm_len, seeds_in_comm2, 20 , 1-10**(-6))

                        num_seeds = np.sum(seeds_in_comm2)
                        print((np.sum(perm == corr)-num_seeds)/(300-num_seeds))
                        seed_i = seed_i +[(np.sum(perm == corr)-num_seeds)/(300-num_seeds)]
          this_one_0_3 = this_one_0_3 + [np.mean(seed_i)]
          this_num_iter = this_num_iter + [np.mean(seed_i_iter)]
    print(this_one_0_3)

# The result being printed is the results of CESSNA for all matrices with correlation 0.3

In [ ]:
# 2. Correlation = 0.6

r = 0.6
for i in range(10):
    graphA = matrices3[(r, i, "A")]
    graphB = matrices3[(r, i, "B")]

    graphA2 = graphA.copy()
    comm_len = [100,100,100]
    this_one_0_6 = []
    this_num_iter = []

    for n_seeds in [1, 5, 10, 20, 50, 75, 100, 150, 200]:
          seed_i = []
          seed_i_iter = []
          seeds_in_comm2 = new_generate_array_comm(comm_len, n_seeds)
          for j in range(0,10):
                        cum_comm = np.cumsum(comm_len)
                        cum_comm = [0] + cum_comm.tolist()
                        perm = np.array(range(0,cum_comm[-1]))


                        for i in range(0, len(seeds_in_comm2)):
                            slice_size = cum_comm[i+1] - (cum_comm[i] + seeds_in_comm2[i])
                            permuted_indices = np.random.permutation(slice_size)
                            shifted_indices = permuted_indices + (cum_comm[i] + seeds_in_comm2[i])
                            perm[(cum_comm[i]+seeds_in_comm2[i]):cum_comm[i+1]] = shifted_indices

                        graphA3 = graphA.copy()
                        graphA2[:] = graphA3[perm][:, perm]
                        corr = block_SGM(graphA2, graphB, comm_len, seeds_in_comm2, 20 , 1-10**(-6))

                        num_seeds = np.sum(seeds_in_comm2)
                        print((np.sum(perm == corr)-num_seeds)/(300-num_seeds))
                        seed_i = seed_i +[(np.sum(perm == corr)-num_seeds)/(300-num_seeds)]
          this_one_0_6 = this_one_0_6 + [np.mean(seed_i)]
          this_num_iter = this_num_iter + [np.mean(seed_i_iter)]
    print(this_one_0_6)

# The result being printed is the results of CESSNA for all matrices with correlation 0.6

In [ ]:
wiki_labels_leiden = np.load('100runs_leiden_wiki_with_singletons_labels.npy', allow_pickle=True)

acc_dict = {}

for runs in range(50):

    indices = wiki_labels_leiden[runs].argsort()

    graphA = wiki_adj['G_EN_Adj'][indices][:, indices]
    graphB = wiki_adj['G_FR_Adj'][indices][:, indices]

    graphA2 = graphA.copy()

    avg_arr = []

    comm_len = [np.sum(wiki_labels_leiden[runs] == x) for x in range(0, max(wiki_labels_leiden[runs])+1)]

    for percentage in range(0,1,1):

          shuffled_perm = shuffle_percentage(list(range(0,1382)), percentage)
          avg = 0
          for num_of_seeds in [0, 5, 50, 150, 250, 382, 500]:
                print("-------------------------------------------------------")
                print("Current number of seeds:" + str(num_of_seeds))
                avg = 0
                seeds_in_comm2 = new_generate_array_comm(comm_len, num_of_seeds)
                for j in range(0,1):
                      cum_comm = np.cumsum(comm_len)
                      cum_comm = [0] + cum_comm.tolist()
                      perm = np.array(range(0,cum_comm[-1]))


                      for i in range(0, len(seeds_in_comm2)):
                          slice_size = cum_comm[i+1] - (cum_comm[i] + seeds_in_comm2[i])
                          permuted_indices = np.random.permutation(slice_size)
                          shifted_indices = permuted_indices + (cum_comm[i] + seeds_in_comm2[i])
                          perm[(cum_comm[i]+seeds_in_comm2[i]):cum_comm[i+1]] = shifted_indices

                      graphA3 = graphA.copy()
                      graphA3[:] = graphA3[shuffled_perm][:, shuffled_perm]
                      graphA2[:] = graphA3[perm][:, perm]
                      #print(perm)
                      #print(len(comm_len))
                      #print(len(seeds_in_comm2))
                      corr = block_SGM(graphA2, graphB, comm_len, seeds_in_comm2, 20 , 1-10**(-6))

                      num_seeds = np.sum(seeds_in_comm2)
                      avg = avg+ ((np.sum(perm == corr)-num_seeds)/(1382-num_seeds))
                      #print((np.sum(perm == corr)-num_seeds)/(279-num_seeds))

                avg_arr = avg_arr + [avg/3]
                #acc_arr_per = acc_arr_per + [avg/10]
                print("avg result for this number of seeds is "+str(avg))
                print("-------------------------------------------------------")
                if num_of_seeds not in acc_dict.keys():
                      acc_dict[num_of_seeds] = [avg]
                else:
                      acc_dict[num_of_seeds].append(avg)
          print("Current acc dict")
          print(acc_dict)


-------------------------------------------------------
Current number of seeds:0
This one uses 20iterations
avg result for this number of seeds is 0.6758321273516642
-------------------------------------------------------
-------------------------------------------------------
Current number of seeds:5
This one uses 20iterations
avg result for this number of seeds is 0.6862745098039216
-------------------------------------------------------
-------------------------------------------------------
Current number of seeds:50
This one uses 19iterations
avg result for this number of seeds is 0.6936936936936937
-------------------------------------------------------
-------------------------------------------------------
Current number of seeds:150
This one uses 20iterations
avg result for this number of seeds is 0.7093117408906883
-------------------------------------------------------
-------------------------------------------------------
Current number of seeds:250
This one uses 15itera

KeyboardInterrupt: 

In [ ]:

import numpy as np
import leidenalg as la
import igraph as ig
from sklearn.cluster import AgglomerativeClustering
from collections import defaultdict
from itertools import groupby

elegans = scipy.io.loadmat('elegansGraph.mat')
graphA = elegans['Achem'].toarray()
graphB = elegans['Agap'].toarray()
graphA2 = graphA.copy()

graph = ig.Graph.Adjacency(graphB.tolist(), mode="undirected")
N = graph.vcount()
n_runs = 100

all_leiden_elegans = []

for i in range(n_runs):
    part = list(la.find_partition(graph, la.ModularityVertexPartition, seed=i))
    all_leiden_elegans.append(part)

#all_leiden_elegans = [np.asarray(part) for part in all_leiden_elegans]
np.save('100runs_leiden_elegans_with_singletons.npy', np.array(all_leiden_elegans, dtype=object), allow_pickle=True)
